# 실험① — 기준선과 교차검증 전제

**이 노트북이 답할 질문**

> 이미지에서 값을 뽑을 때, **"두 모델이 다르면 틀린 것이다"** 라는 전제가 참인가?

이 프로젝트의 설계 전체가 그 전제 위에 서 있다. 참이 아니면 검증 계층을 다시 짜야 한다.

---

## 왜 노트북에서 먼저 하는가

`md-rag-chatbot` 에서 배운 것 — **코드에 상수를 박기 전에 재야 한다.**
거기서 임계값 `0.6` 과 `task_type` 분기는 "그럴듯해서" 들어간 숫자였고, 재보니 둘 다 틀렸다.

노트북은 **틀린 경로도 남긴다.** 최종 답만 본 코드로 넘어간다.

---

## 재는 대상

| | 모델 | 벤더 | 성격 |
|---|---|---|---|
| A | `gemini-2.5-flash` | Google | 상용 폐쇄 가중치 |
| B | `qwen/qwen3.8-27b` | Alibaba 가중치 · Groq 서빙 | 오픈소스 가중치 |

계열이 달라야 **같은 실수를 할 확률이 낮다.** 같은 모델의 크기만 다른 두 개로 갔다면
교차검증의 전제부터 위태로웠다.

---

## 단계

| | 무엇을 | 왜 |
|---|---|---|
| **Phase 0** | 데이터 로드 + **지표 정의** | `md-rag-chatbot` 에서 가장 많이 틀린 자리가 지표였다 |
| Phase 1 | 두 모델 기준선 정확도 | 무엇이 좋은지 알아야 다음이 있다 |
| **Phase 2** | **교차검증 전제 검증** | 설계의 전제. 여기가 이 노트북의 목적 |
| Phase 3 | 코드 검산 검출력 | CORD · 영양성분표 필요 |
| Phase 4 | 자동/보류 판정 기준 | `md-rag-chatbot` 의 `RERANK_THRESHOLD` 에 해당 |

**오늘은 Phase 0 까지.**

---

# Phase 0 — 지표부터 정한다

**먼저 숫자를 뽑고 싶은 유혹을 참는 자리다.**

`md-rag-chatbot` 실험① 에서 *분리도(평균의 차이)* 로 재다가 "검색이 나쁘다" 는 오판을 했다.
평균은 분포의 겹침을 못 본다. 지표가 틀리면 결론도 틀린다 — 그때는 그걸 몰랐다.

여기서 미리 정해야 할 것 세 가지.

### ① 필드 하나가 "맞았다" 는 무엇인가

```
정답      ₩1,000
모델 A    1000
모델 B    1,000원
```

셋 다 같은 값인데 문자열로는 전부 다르다. **완전일치로 재면 정확도가 실제보다 낮게 나온다.**
→ 필드 종류별로 **정규화 후 비교**한다. 금액은 숫자로, 날짜는 ISO 로, 상호명은 공백·대소문자 정리.

**★ 그런데 정규화를 세게 하면 진짜 오류까지 덮는다.** 이 경계를 어디에 둘지가 Phase 0 의 숙제다.

### ② 필드마다 자가 다른데 어떻게 합치나

금액은 숫자 비교, 상호명은 문자열 비교다. 단순 평균을 내면 **어느 필드가 중요한지가 사라진다.**
→ 필드별 정확도를 따로 보고, 합칠 때는 가중을 명시한다.

### ③ "교차검증이 잘 작동한다" 를 하나의 숫자로

이게 제일 어렵다. Phase 2 에서 나올 네 칸이 답을 정한다.

```
                    실제로 맞음      실제로 틀림
두 모델 일치          ✅ 자동처리       ❌ 못 잡는다  ← 이 비율이 설계의 상한
두 모델 불일치        ⚠️ 헛경보         ✅ 잡았다
```

**우하단이 크고 우상단이 작으면** 교차검증이 쓸모 있다.
**우상단이 크면** 교차검증만으로는 부족하고 코드 검산이 더 중요해진다.

→ 재기 전에는 모른다. 그래서 이 노트북이 있다.

In [1]:
import os, sys, json, time, base64, io, re
from pathlib import Path

# ★ 출력 정리 — 원인을 알고 끄는 것이므로 남긴다 (md-rag-chatbot 에서 같은 처리를 했다)
import warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message=".*IProgress.*")

DATA = Path("data"); DATA.mkdir(exist_ok=True)
CACHE = DATA / "calls.pkl"          # 모델 호출 결과 캐시 — Groq 가 4장/분이라 필수

# ★ 키를 찾는 순서 — 환경변수 → 이 저장소의 .env → EXTRA_ENV_FILE 로 지정한 파일
#   개인 경로를 코드에 박지 않는다. 다른 저장소의 .env 를 재사용하려면 노트북을 띄울 때
#   EXTRA_ENV_FILE=~/어딘가/.env jupyter lab  처럼 넘긴다.
def env(name: str) -> str:
    if v := os.environ.get(name):
        return v
    paths = [Path("../backend/.env"), Path("../.env")]
    if extra := os.environ.get("EXTRA_ENV_FILE"):
        paths.append(Path(extra).expanduser())
    for path in paths:
        if path.exists() and (m := re.search(rf"^{name}=(.+)$", path.read_text(), re.M)):
            return m.group(1).strip().strip('"\'')
    raise RuntimeError(f"{name} 를 찾을 수 없습니다. backend/.env 에 넣으세요.")

GEMINI_KEY, GROQ_KEY = env("GEMINI_API_KEY"), env("GROQ_API_KEY")
MODEL_A, MODEL_B = "gemini-2.5-flash", "qwen/qwen3.8-27b"
print(f"모델 A  {MODEL_A}\n모델 B  {MODEL_B}")
print(f"키      Gemini {'있음' if GEMINI_KEY else '없음'} · Groq {'있음' if GROQ_KEY else '없음'}")

모델 A  gemini-2.5-flash
모델 B  qwen/qwen3.8-27b
키      Gemini 있음 · Groq 있음


## 0-1. 한도 — 실험 규모를 정하는 값

오늘 실측한 것. **Groq 의 분당 토큰이 병목이다.**

| | 이미지 1장 | 한도 | 처리량 |
|---|---|---|---|
| Gemini | 470 토큰 | 1,500 요청/일 | 여유 |
| **Groq** | **1,873 토큰** | **8,000 토큰/분** | **약 4장/분** |

200장 교차검증이면 **약 50분.** 그래서 **호출 결과를 반드시 캐시**한다.
`md-rag-chatbot` 의 `embed_cache.pkl` 과 같은 이유 — 노트북을 다시 돌릴 때마다 50분을 쓸 수 없다.

---

### ⚠️ 2026-09-07 정정 — Gemini 는 1,500 이 아니라 **하루 20 요청**이다

위 표의 `1,500 요청/일` 이 **75배 틀렸다.** 200행 배치를 돌리자 28회째부터 429 가 났고
오류 본문에 한도가 그대로 있었다.

```
quotaId:    GenerateRequestsPerDayPerProjectPerModel-FreeTier
model:      gemini-2.5-flash
quotaValue: 20
```

**왜 틀렸나 — 실측한 줄 알았는데 아니었다.**
9/06 에 호출은 실제로 했다. 그런데 잰 것은 *"1회 호출이 되는가"* 와 *"토큰이 몇 개인가"* 였고,
**하루 한도는 재지 않고 문서에서 읽은 숫자를 옆에 나란히 적었다.**
바로 위에 *"공식 문서는 숫자를 안 적으니 직접 호출해서 확정했다"* 고 써놓고서.

> **한 번 호출된다고 한도를 안 것이 아니다. 한도는 한도에 부딪혀야 알 수 있다.**
> 재지 않은 것을 잰 것 옆에 적으면 **둘 다 실측처럼 보인다.**
> `0-8` 의 *"바닥이라 이름 붙인 숫자가 사실은 내 코드의 한계였다"* 와 같은 종류다.

**실제 한도**

| | 요청/일 | 분당 | 200행 |
|---|---|---|---|
| Gemini 2.5-flash | **20** | — | **10일** |
| **Groq qwen3.8-27b** | **1,000** | ITPM **7,000** · TPM 8,000 | **50~67분** |

이미지 1장 입력이 Groq 866~1,890 토큰이라 **ITPM 7,000 이 분당 4~8장으로 묶는다.**
67분은 **한도를 지키려 기다리는 시간이지 비용이 아니다.**

### → 모델 하나로 간다: Groq `qwen/qwen3.8-27b`

무료 티어 안에서 200행을 하루에 끝낼 수 있는 유일한 쪽이다.

**잃는 것 — 교차검증(Phase 2).** 이 노트북의 원래 목적이다. **보류이지 폐기가 아니다.**
**메우는 것 — 검산 계층.** `0-8` 에서 정답 라벨 검산을 100% 까지 올려놨으므로,
그 규칙 위에서는 **모델 출력의 검산 실패가 곧 진짜 신호**다.
교차검증은 모델 둘이 필요하지만 **검산은 하나면 된다.**

In [2]:
# 한도를 헤더로 다시 확인 — 계정 상태가 바뀌었을 수 있다
import httpx
r = httpx.post("https://api.groq.com/openai/v1/chat/completions",
               headers={"Authorization": f"Bearer {GROQ_KEY}"},
               json={"model": MODEL_B, "max_tokens": 5,
                     "messages": [{"role": "user", "content": "hi"}]}, timeout=30)
lim = {k: v for k, v in r.headers.items() if "ratelimit" in k.lower()}
for k, v in sorted(lim.items()):
    print(f"  {k:34} {v}")

tpm = int(lim.get("x-ratelimit-limit-tokens", 0))
print(f"\n이미지 1장 ≈ 1,873 토큰 → 분당 {tpm // 1873 if tpm else '?'}장")
print(f"200장 교차검증 ≈ {200 / (tpm/1873):.0f}분" if tpm else "")

  x-ratelimit-limit-requests         1000
  x-ratelimit-limit-tokens           8000
  x-ratelimit-remaining-requests     981
  x-ratelimit-remaining-tokens       7982
  x-ratelimit-reset-requests         27m21.6s
  x-ratelimit-reset-tokens           134ms

이미지 1장 ≈ 1,873 토큰 → 분당 4장
200장 교차검증 ≈ 47분


## 0-2. 데이터 — SROIE

> ### ⚠️ 2026-09-07 정정 — 이 절의 결론은 뒤집혔다
> 아래 표의 라이선스 두 칸이 **둘 다 틀렸다.** 지우지 않고 남긴다 — 어떻게 틀렸는지가
> 이 노트북의 결과물이다. 정정된 판단은 **`0-2b`** 에 있다.

**정답 라벨이 있고 상업적 사용이 가능한** 유일한 선택지다.

| 데이터셋 | 라이선스 | 판단 |
|---|---|---|
| ~~**SROIE**~~ | ~~**MIT**~~ → **미표기** | ~~채택~~ → **제외** |
| ~~CORD~~ | ~~CC BY-SA 4.0~~ → **CC BY 4.0** | ~~Phase 3 에서 검토~~ → **채택** |
| FUNSD | 비상업 학술 전용 | **제외** — 이 저장소는 공개물이라 충돌 여지 |

### ★ 설계를 세운 뒤에 발견한 것

SROIE 의 필드는 **`company` · `address` · `total` · `date` 넷뿐이고 품목 목록이 없다.**

```
검산 ① 칼로리 = 탄단지 환산    영양성분표에만
검산 ② 1회 제공량 × 횟수       영양성분표에만
검산 ③ 부분 ≤ 전체             품목 필요
       합계 = 품목합           SROIE 엔 품목이 없어 불가
```

**즉 SROIE 로는 "얼마나 정확히 읽나" 는 재지만 "검산이 오류를 잡나" 는 못 잰다.**
검증 계층이 이 프로젝트의 핵심인데 그걸 잴 데이터가 없다.

→ Phase 3 은 CORD 또는 직접 촬영한 영양성분표가 필요하다. ~~**Phase 0~2 는 SROIE 로 충분하다.**~~

★ 이 지적은 여전히 옳다. 그리고 `0-2b` 의 라이선스 대조가 **같은 방향을 가리켰다** —
두 개의 독립적인 이유가 CORD 를 지목한 셈이다.

---

## 0-2b. ★ 정정 — 라이선스를 원본에서 대조했더니 순위가 뒤집혔다

바로 아래 셀에 *"미러를 쓸 때는 라이선스 표기를 원본과 대조한다"* 고 써뒀다.
**대조해보니 우리가 적어둔 `MIT` 자체가 그 함정이었다.**

### MIT 는 어디서 온 것인가

`github.com/zzzDavid/ICDAR-2019-SROIE` 에 MIT LICENSE 파일이 있다. 그런데:

| | |
|---|---|
| MIT 가 덮는 것 | **그 저장소의 코드** — "this software and associated files" |
| 그 저장소에 데이터가 있나 | **없다.** 구글드라이브/Baidu 링크 안내 + 라벨 오류 수정 스크립트뿐 |
| 데이터셋 라이선스 언급 | **없다** |

즉 **대회 참가팀 코드의 라이선스를 데이터셋 라이선스로 읽었다.**

### 원본 확인 — 어디에도 표기가 없다

| 출처 | 라이선스 표기 |
|---|---|
| 공식 RRC 페이지 `rrc.cvc.uab.es/?ch=13` | **없음** (다운로드는 포털 로그인 뒤) |
| 대회 논문 arXiv 2103.10213 | **없음** — CC BY 4.0 은 **논문 문서**에 붙은 것 |
| HF `rth/sroie-2019-v2` · `darentang/sroie` · `Metric-AI/icdar_sroie` | **전부 미표기** |
| HF `jsdnrs/ICDAR2019-SROIE` | CC-BY-4.0 — 업로더 자기 선언, 원본 근거 없음 |

**SROIE 는 금지된 게 아니라 "알 수 없는" 데이터셋이다.** FUNSD 를 뺀 기준을 그대로 적용하면
SROIE 는 애초에 그 심사대에 오른 적조차 없다 — 근거 문서를 본 적이 없기 때문이다.

### 그리고 CORD 는 우리가 적은 것보다 조건이 가볍다

| | 우리가 적어둔 것 | 실제 |
|---|---|---|
| CORD | CC BY-**SA** 4.0 (출처표기 + 동일조건) | **CC BY 4.0 — 출처표기만** |

원본 `clovaai/cord` README 첫 줄 배지와 HF `naver-clova-ix/cord-v2` 메타데이터가 **일치한다.**
SA 를 잘못 붙여서 CORD 를 실제보다 무거운 선택지로 취급하고 있었다.

### 결론 — CORD 단독

|  | 라이선스 근거 | 품목(Phase 3) | 필드 |
|---|---|---|---|
| **CORD** | **명확 · 원본↔미러 일치** | **있다** | 4범주 30종 |
| SROIE | 없음 | 없다 | 4개 |

**능력이 포함관계다.** CORD 에서 `total`·`date` 만 뽑으면 SROIE 수준의 단순 실험이 그대로 되고,
반대는 안 된다. 게다가 CORD 를 쓰면 Phase 3 이 **영양성분표 촬영을 기다리지 않고** 시작된다.

> ★ 배운 것: 라이선스는 **미러가 아니라 원본에서** 확인한다. 그리고 저장소에 붙은 라이선스가
> 덮는 대상이 **코드인지 데이터인지** 구분한다. Groq 의 `403` 을 모델 탓으로 읽을 뻔한 것과
> 같은 종류의 오독이다 — **표기를 봤지 대상을 안 봤다.**

### 덤 — 지표 설계가 오히려 제대로 된다

`0-1` 의 숙제 ②(*필드마다 성격이 다른데 어떻게 합치나*)는 SROIE 4필드에선 잘 드러나지 않는다.
CORD 의 30종 엔티티에서는 피할 수 없다. 본 코드가 **임의 스키마를 받는 엔진**인 이상
어려운 쪽에서 지표를 정하는 게 맞다.

---

## 0-3. CORD 로드 — 무엇을 받고 무엇을 안 받는가

**받을 것: `test` 100 + `validation` 100 = 200행 (약 476MB).**
`train` 800행(1.97GB)은 **받지 않는다.** 파인튜닝을 안 하므로 train/test 구분이 우리에겐
의미가 없고, Groq 이 4장/분이라 **200장이 이미 50분**이다. 더 받아도 못 돌린다.

### ⚠️ 실측 정정 — `split="test"` 는 train 다운로드를 막지 못한다

처음엔 `load_dataset(..., split="test")` 로 짰다. **틀렸다.** 돌려보니 `Generating train split`
이 뜨면서 캐시가 6.9GB 가 됐다. `split=` 은 **받은 뒤에 고르는** 인자이지 받을 것을 고르는
인자가 아니다 — 이 데이터셋 카드에 split 선언이 없어서 `datasets` 가 `data/` 폴더를
통째로 글롭한다.

```
data/train-00000-of-00004…parquet   490MB   ┐
data/train-00001-of-00004…parquet   441MB   │  split="test" 여도
data/train-00002-of-00004…parquet   444MB   │  전부 받는다
data/train-00003-of-00004…parquet   456MB   ┘
data/test-00000-of-00001…parquet    234MB   ← 실제로 필요한 것
data/validation-00000-of-00001…parquet 242MB
```

→ `data_files` 로 **파일을 직접 지정**해야 한다. 그러면 `dataset_info` 가 기대하는 train 이
없다고 `ExpectedMoreSplitsError` 가 나므로 `verification_mode="no_checks"` 를 같이 준다.

★ *"문서로 논쟁하지 않고 직접 호출해서 확정한다"* 를 또 한 번 확인한 자리다.
Gemini 한도 때도 같았다.

### `valid_line` 은 버리지 않는다

`ground_truth` 안에는 우리가 쓸 `gt_parse`(정답 필드) 말고 `valid_line` 이 같이 들어 있다.
bbox + 텍스트인데, **사람이 검수한 것이라 사실상 "완벽한 OCR 결과"** 다.

이 프로젝트는 OCR 엔진을 기본 경로에 두지 않는다 — 시스템 설치가 필요해서
*"클론해서 API 키만 넣으면 동작"* 조건을 깬다. 하지만 그건 **배포 제약이지
"OCR 이 쓸모없다"는 근거가 아니다.**

```
OCR  →  "3S,OOO"   깨진다      → 눈에 띈다
VLM  →  "38,000"   지어낸다    → 안 띈다
```

실패 양상이 반대라는 건 교차검증 관점에서 **독립적**이라는 뜻이다.
어쩌면 VLM 두 개보다 **VLM + OCR 이 더 독립적**일 수 있다.

`valid_line` 을 들고 있으면 **엔진 설치 0으로** 이걸 잴 수 있다 —
*"OCR 을 세 번째 검증자로 넣으면 오류를 얼마나 더 잡는가"* 의 **상한**.
상한이 낮으면 OCR 트랙을 접을 근거가 되고, 높으면 `[local]` extra 로 넣을 근거가 된다.
어느 쪽이든 **"안 쓴다" 를 재고 나서 말하게 된다.**

→ 지금 버리면 나중에 478MB 를 다시 받아야 한다. 파싱 함수 하나 더 두는 것 외엔 비용이 없다.

In [3]:
# CORD 내려받기 — 출처·라이선스를 코드에 남긴다
#
# CORD: A Consolidated Receipt Dataset for Post-OCR Parsing
#   Park, Shin, Lee, Lee, Surh, Seo, Lee (NAVER CLOVA) · DI Workshop @ NeurIPS 2019
#   원본 https://github.com/clovaai/cord
#   배포 https://huggingface.co/datasets/naver-clova-ix/cord-v2
#   라이선스 CC BY 4.0 — 출처표기 + 변경사항 명시 (README "데이터" 절)
#
# ★ 라이선스는 원본에서 확인했다. 미러의 표기를 믿지 않는다 (0-2b 참고)
from datasets import load_dataset, concatenate_datasets

# ★ split= 이 아니라 data_files= 다. split= 은 train 까지 받은 뒤에 고른다 (위 정정 참고)
_d = load_dataset(
    "naver-clova-ix/cord-v2",
    data_files={"test": "data/test-*.parquet", "validation": "data/validation-*.parquet"},
    verification_mode="no_checks",   # dataset_info 가 train 을 기대하므로 검증을 끈다
)
ds = concatenate_datasets([_d["test"], _d["validation"]])

# ground_truth 는 dict 가 아니라 JSON "문자열" 이다 (HF 스키마상 Value(string))
def label(row):
    return json.loads(row["ground_truth"])["gt_parse"]     # 정답 필드 — 지금 쓰는 것

def ocr_lines(row):
    return json.loads(row["ground_truth"])["valid_line"]   # 사람이 검수한 OCR = "완벽한 OCR"
                                                           # 지금은 안 쓴다. Phase 2 상한 측정용

GT = [label(r) for r in ds]        # 라벨만 미리 파싱해둔다 (200개, 가볍다)
print(f"{len(ds)}행 · 컬럼 {ds.column_names} · 첫 이미지 {ds[0]['image'].size}")
print(json.dumps(GT[0], ensure_ascii=False)[:200])

200행 · 컬럼 ['image', 'ground_truth'] · 첫 이미지 (432, 648)
{"menu": {"nm": "-TICKET CP", "num": "901016", "cnt": "2", "price": "60.000", "itemsubtotal": "60.000"}, "sub_total": {"subtotal_price": "60.000", "discount_price": "-60.000", "tax_price": "5.455"}, "


---

## 0-4. 라벨 평탄화 — 실측한 함정 셋을 흡수한다

정답 라벨을 **품목 리스트**로 펴는 자리다. 단순해 보이지만 200행을 재보니
**예외 없이 조용히 틀리는** 자리가 셋 있었다.

| | 실측 (200행) | 그냥 짰다면 |
|---|---|---|
| ① `menu` 타입 | **dict 85 · list 115** | `for it in menu` 가 dict 의 **키 문자열**을 순회. 예외 안 남 |
| ② `sub_total` 부재 | `[menu, sub_total, total]` 133 · **`[menu, total]` 67** | 없는 것을 0 으로 읽으면 검산이 조용히 통과 |
| ③ 필드 등장률 | `nm` 100% · `price` 99% · `cnt` 91% · **`unitprice` 25%** · **`sub` 9%** | 전부 필수 스키마로 만들면 대부분 파싱 실패 |

### ① 은 왜 위험한가

품목이 하나면 CORD 는 리스트가 아니라 **dict 하나**로 준다.

```python
menu = {"nm": "TICKET", "price": "60.000"}
for it in menu:          # it 은 "nm", "price" — 문자열이다
    it["price"]          # TypeError? 아니다. 문자열 인덱싱이라 "nm"[...] 로 조용히 이상해진다
```

200행 중 **85행(42.5%)** 이 dict 다. 절반 가까이가 조용히 틀린다.

### ② 는 "적용 불가" 를 만든다

`sub_total` 이 통째로 없는 행이 **67행**, `sub_total` 은 있는데 `subtotal_price` 가 없는 행이
**1행** — 합쳐 **68/200 (34%)**. 여기서 합계 검산은 **틀린 게 아니라 못 하는** 것이다.

```python
gt.get("sub_total", {}).get("subtotal_price", 0)   # ← 이렇게 짜면 안 된다
```

**없음(None) 과 0원이 같아진다.** 그러면 품목합이 0이 아닌 한 검산은 "불일치"로 뜨고,
검출력을 재려는 실험이 **헛경보만 세게 된다.**

→ 설계의 `CheckResult.passed: bool | None` 에서 `None`(적용 불가)을 둔 게 옳았다.
**이건 설계 때 추측으로 넣은 필드였는데 실측이 근거를 줬다.**

### ③ 중 `sub` 는 판단이 필요하다

품목 안에 품목이 중첩된다 (`sub`, 200행에 41개).

```json
{"nm": "JASMINE MT (L)", "cnt": "1", "price": "24,000",
 "sub": {"nm": "COCONUT JELLY (L)", "price": "4,000"}}
```

이 영수증의 `subtotal_price` 는 **28,000** 이다. 즉 **하위 품목도 합계에 들어간다.**
눈으로 한 건 본 것이니 가정이다 — **`0-6` 에서 200행으로 재서 확정한다.**

In [4]:
def as_list(x):
    """CORD 는 원소가 하나면 list 가 아니라 그 자체를 준다. 항상 list 로 만든다.
       ★ 200행 중 85행이 dict 다 (0-4 ① 참고)"""
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


def items_of(gt, keep_sub=True):
    """gt_parse → 품목 리스트. keep_sub 는 중첩 하위품목(sub)을 펼칠지 여부.
       ★ 어느 쪽이 맞는지는 0-6 에서 실측으로 정한다"""
    out = []
    for it in as_list(gt.get("menu")):
        if not isinstance(it, dict):
            continue
        out.append({k: v for k, v in it.items() if k != "sub"})
        if keep_sub:
            for s in as_list(it.get("sub")):
                if isinstance(s, dict):
                    out.append({**s, "_sub_of": it.get("nm")})
    return out


def subtotal_of(gt):
    """subtotal_price 를 꺼낸다. ★ 없으면 0 이 아니라 None 이다 (0-4 ② 참고)
       없음(적용 불가)과 0원을 섞으면 검출력 측정이 무너진다"""
    st = gt.get("sub_total")
    if not isinstance(st, dict):
        return None
    return st.get("subtotal_price")


# 실측 확인
import collections
print("menu 타입      :", dict(collections.Counter(type(g.get("menu")).__name__ for g in GT)))
print("최상위 키 조합 :", {"+".join(sorted(k)): v for k, v in
      collections.Counter(tuple(sorted(g)) for g in GT).most_common()})
print("subtotal 없음  :", sum(1 for g in GT if subtotal_of(g) is None), "행")
print("품목 총계      :", sum(len(items_of(g)) for g in GT),
      "(sub 미포함 시", sum(len(items_of(g, keep_sub=False)) for g in GT), ")")

menu 타입      : {'dict': 85, 'list': 115}
최상위 키 조합 : {'menu+sub_total+total': 133, 'menu+total': 67}
subtotal 없음  : 68 행
품목 총계      : 558 (sub 미포함 시 472 )


---

## 0-5. ★ 금액 정규화 — 여기가 1000배가 걸린 자리다

`0-1` 에 이렇게 적어뒀다:

> **★ 그런데 정규화를 세게 하면 진짜 오류까지 덮는다.** 이 경계를 어디에 둘지가 Phase 0 의 숙제다.

**재보니 위험의 방향이 반대인 쪽이 더 크다.**

### 왜 — 인도네시아는 점(`.`)이 천단위 구분자다

| 형식 | 예 | 실제 값 |
|---|---|---|
| 쉼표 = 천단위 | `24,000` | 24,000 |
| **점 = 천단위** | **`60.000`** | **60,000** ← 60.0 이 아니다 |
| 구분자 없음 | `17500` | 17,500 |
| 통화기호 혼재 | `Rp 3.000` · `Rp.56.000` | 3,000 · 56,000 |
| 소수점 (수량·세금) | `2.00` · `48982.68` | 2 · 48,982.68 |

```
float(s.replace(",", ""))       "24,000" → 24000     맞음
                                "60.000" → 60.0      ★ 1000배 틀림

re.sub(r"[.,]", "", s)          "60.000" → 60000     맞음
                                "2.00"   → 200       ★ 수량이 100배 틀림
```

**정규화를 잘못하면 오류를 덮는 게 아니라 없던 오류를 만든다.**
그리고 금액 파싱은 합계 검산의 **전제조건**이라, 파서가 틀리면
우리는 "검산 검출력" 이 아니라 **우리 파서의 버그**를 재게 된다.

### 후보 셋

| | 규칙 | 예상 약점 |
|---|---|---|
| **P1** | 쉼표만 제거 | `60.000` 계열에서 1000배 |
| **P2** | 구분자 전부 제거 | 진짜 소수점(`2.00`)에서 100배 |
| **P3** | **마지막 구분자 뒤 자릿수로 판정** — 3자리면 천단위, 1~2자리면 소수점 | 루피아는 소수를 거의 안 쓰므로 위험 낮음 |

어느 것이 맞는지 **눈으로 정하지 않는다.** `0-6` 에서 잰다.

In [5]:
import re

def _clean(s):
    """숫자·구분자·부호만 남긴다. 'Rp 3.000' '@120,000' '(0)' '40,000.' 같은 것들 처리"""
    if isinstance(s, list):          # 값이 리스트로 오는 경우가 있다 (["46.636", "46.636"])
        s = s[0] if s else None
    if s is None:
        return None
    if isinstance(s, (int, float)):
        return str(s)
    t = re.sub(r"[^\d.,\-]", "", str(s)).strip().rstrip(".,")
    return t if re.search(r"\d", t) else None


def P1(s):
    """후보 1 — 쉼표만 제거 (가장 순진한 방법)"""
    t = _clean(s)
    try:
        return float(t.replace(",", "")) if t else None
    except ValueError:
        return None


def P2(s):
    """후보 2 — 구분자를 전부 제거"""
    t = _clean(s)
    try:
        return float(re.sub(r"[.,]", "", t)) if t else None
    except ValueError:
        return None


def P3(s):
    """후보 3 — 마지막 구분자 뒤 자릿수로 판정
       뒤가 3자리면 천단위 구분자, 1~2자리면 소수점"""
    t = _clean(s)
    if not t:
        return None
    neg = t.startswith("-")
    t = t.lstrip("-")
    seps = [m.start() for m in re.finditer(r"[.,]", t)]
    try:
        if not seps:
            v = float(t)
        else:
            pos = seps[-1]
            tail = len(t) - pos - 1
            if tail in (1, 2):
                v = float(re.sub(r"[.,]", "", t[:pos]) + "." + t[pos + 1:])
            else:
                v = float(re.sub(r"[.,]", "", t))
    except ValueError:
        return None
    return -v if neg else v


for s in ["24,000", "60.000", "17500", "Rp 3.000", "Rp.56.000", "2.00", "48982.68", "-60.000"]:
    print(f"  {s:12} P1={str(P1(s)):12} P2={str(P2(s)):12} P3={P3(s)}")

  24,000       P1=24000.0      P2=24000.0      P3=24000.0
  60.000       P1=60.0         P2=60000.0      P3=60000.0
  17500        P1=17500.0      P2=17500.0      P3=17500.0
  Rp 3.000     P1=3.0          P2=3000.0       P3=3000.0
  Rp.56.000    P1=None         P2=56000.0      P3=56000.0
  2.00         P1=2.0          P2=200.0        P3=2.0
  48982.68     P1=48982.68     P2=4898268.0    P3=48982.68
  -60.000      P1=-60.0        P2=-60000.0     P3=-60000.0


---

## 0-6. ★★ 정답 라벨의 내부 산수로 파서를 고른다

파서가 맞는지 **어떻게 아는가.** 눈으로 보는 건 47개 값을 놓친다(아래에서 확인).

**영수증은 스스로를 검산할 수 있다.** 정답 라벨에서

```
Σ(품목 price)  ≈  subtotal_price
```

가 성립해야 한다. 파서가 어떤 값에서 1000배 틀리면 이 등식이 깨진다.
**즉 이 프로젝트가 엔진에 넣으려는 바로 그 검산으로, 우리 파서를 먼저 검증한다.**

덤으로 두 가지가 같이 나온다.
1. `0-4` 에서 미룬 질문 — **`sub` 하위품목을 합계에 넣어야 하는가**
2. **검산 검출력의 상한** — 정답 라벨에서조차 이 검산이 성립하지 않는 비율

In [6]:
def check_subtotal(parser, keep_sub=True, tol=0.005):
    """정답 라벨에서 Σ(품목 price) ≈ subtotal_price 가 성립하는가.
       반환: (성립, 불성립, 적용불가, 불성립 행의 배율 목록)
       ★ 적용불가를 불성립으로 세지 않는 것이 핵심이다 (0-4 ② 참고)"""
    ok = bad = na = 0
    ratios = []
    for g in GT:
        st = subtotal_of(g)
        if st is None:
            na += 1
            continue
        sv = parser(st)
        prices = [parser(it["price"]) for it in items_of(g, keep_sub) if "price" in it]
        if sv is None or sv == 0 or not prices or any(p is None for p in prices):
            na += 1
            continue
        s = sum(prices)
        if abs(s - sv) / abs(sv) <= tol:
            ok += 1
        else:
            bad += 1
            ratios.append(round(s / sv, 3))
    return ok, bad, na, ratios


print("검산: Σ(품목 price) ≈ subtotal_price   (허용오차 0.5%)\n")
print(f"  {'파서':26} {'성립':>4} {'불성립':>6} {'적용불가':>8}  성립률")
for name, P in [("P1 쉼표만 제거", P1), ("P2 구분자 전부 제거", P2), ("P3 뒤 자릿수 판정", P3)]:
    ok, bad, na, _ = check_subtotal(P)
    print(f"  {name:26} {ok:4} {bad:6} {na:8}  {ok/(ok+bad)*100:5.1f}%")

print("\nsub 하위품목을 합계에 넣어야 하는가 (파서=P3)")
for keep in (True, False):
    ok, bad, na, _ = check_subtotal(P3, keep_sub=keep)
    print(f"  keep_sub={str(keep):5}  성립 {ok:3} / 불성립 {bad:3}  →  {ok/(ok+bad)*100:5.1f}%")

검산: Σ(품목 price) ≈ subtotal_price   (허용오차 0.5%)

  파서                           성립    불성립     적용불가  성립률
  P1 쉼표만 제거                   112     18       70   86.2%
  P2 구분자 전부 제거                122      9       69   93.1%
  P3 뒤 자릿수 판정                 122      9       69   93.1%

sub 하위품목을 합계에 넣어야 하는가 (파서=P3)
  keep_sub=True   성립 122 / 불성립   9  →   93.1%
  keep_sub=False  성립 118 / 불성립  13  →   90.1%


### 결과 — 그리고 ★ 검산이 파서를 증명하지 못한다

```
파서                        성립   불성립   적용불가   성립률
P1 쉼표만 제거               112      18       70      86.2%
P2 구분자 전부 제거           122       9       69      93.1%
P3 뒤 자릿수 판정             122       9       69      93.1%
```

**P1 은 6.9%p 떨어진다.** 그 18행이 `60.000` 계열에서 1000배 틀린 것들이다.
순진한 파서가 **200행 중 18행을 조용히 망가뜨린다**는 게 숫자로 나왔다.

`sub` 질문도 답이 나왔다.

```
keep_sub=True    성립 122 / 불성립  9   →  93.1%      ← 채택
keep_sub=False   성립 118 / 불성립 13   →  90.1%
```

**하위 품목은 합계에 들어간다.** 눈으로 본 한 건이 200행에서 3.0%p 로 확인됐다.

---

### ★★ 그런데 P2 와 P3 가 동점이다. 그리고 P2 는 틀렸다.

이게 이 절에서 제일 중요한 대목이다.

```
price        '30303.00'    P2 = 3,030,300     P3 = 30,303      ← P2 가 100배 틀림
menuqty_cnt  '2.00'        P2 = 200           P3 = 2
total_price  '365000.00'   P2 = 36,500,000    P3 = 365,000
```

**고유 문자열 1,459개 중 47개에서 갈리는데, 전부 P2 가 틀렸다.**
그런데 검산 성립률은 **똑같이 93.1%** 로 나온다.

왜인가 — `30303.00` 형식을 쓰는 영수증은 **품목도 subtotal 도 같은 형식**이다.
그래서 P2 는 양변을 똑같이 100배로 부풀리고, **비율 비교에서 상쇄된다.**

> ### 이 프로젝트의 주제가 그대로 되돌아왔다
>
> 우리는 *"VLM 이 그럴듯한 오답을 쓴다"* 를 잡으려고 검산을 넣는다.
> 그런데 **검산 자체가 그럴듯한 오답을 통과시켰다.**
> **일관된 배율 오차는 비율 검산으로 못 잡는다.**
>
> → 검산은 **틀린 것을 잡지, 맞은 것을 증명하지 않는다.**
> `confidence.judge()` 가 "검산 통과 = 자동처리" 로 가면 이 구멍을 그대로 물려받는다.
> **검산 통과는 근거가 아니라 반증 실패다.**

그래서 P3 를 고른 근거는 검산 성립률이 아니라 **[F] 에서 값 단위로 대조한 47건**이다.
지표 하나만 보고 골랐으면 P2 를 골랐을 것이고, 수량·세금 필드가 100배로 깨졌을 것이다.
`md-rag-chatbot` 실험① 에서 *분리도* 하나만 보고 오판한 것과 같은 구조다.

In [7]:
# ★ P2 와 P3 가 실제로 갈리는 값을 전부 뽑는다 — 성립률로는 안 보이는 차이
def all_strings():
    for g in GT:
        for it in items_of(g):
            for k, v in it.items():
                if isinstance(v, str):
                    yield k, v
        for sec in ("sub_total", "total"):
            d = g.get(sec)
            if isinstance(d, dict):
                for k, v in d.items():
                    if isinstance(v, str):
                        yield k, v

uniq = collections.Counter(all_strings())
diff = [(k, v, P2(v), P3(v), n) for (k, v), n in uniq.items() if P2(v) != P3(v)]
print(f"고유 문자열 {len(uniq)}개 중 P2≠P3 인 것: {len(diff)}개\n")
for k, v, a, b, n in sorted(diff, key=lambda x: -x[4])[:10]:
    print(f"  {k:14} {v!r:14} P2={str(a):14} P3={b}")

고유 문자열 1459개 중 P2≠P3 인 것: 47개

  price          '30303.00'     P2=3030300.0      P3=30303.0
  menuqty_cnt    '2.00'         P2=200.0          P3=2.0
  price          '47619.00'     P2=4761900.0      P3=47619.0
  price          '12987.00'     P2=1298700.0      P3=12987.0
  menuqty_cnt    '1.00xITEMs'   P2=100.0          P3=1.0
  menuqty_cnt    '1.00'         P2=100.0          P3=1.0
  price          '34632.00'     P2=3463200.0      P3=34632.0
  price          '43290.00'     P2=4329000.0      P3=43290.0
  price          '25974.00'     P2=2597400.0      P3=25974.0
  subtotal_price '316017.00'    P2=31601700.0     P3=316017.0


---

## 0-7. ★ 검산 검출력의 상한 — 설계를 건드리는 숫자

`confidence.judge()` 의 초안 가설은 `code_design.md` 에 이렇게 적혀 있다.

```
검산 실패가 하나라도 있으면            → 보류
교차검증 불일치가 핵심 필드에 있으면     → 보류
그 외                                → 자동
```

**이 가설을 정답 라벨에 걸어보면 어떻게 되는가.** 정답인데 보류되면 그게 헛경보의 바닥이다.

In [8]:
ok, bad, na, ratios = check_subtotal(P3)
tot = len(GT)
print(f"전체 {tot}행")
print(f"  적용 불가   {na:3}행 ({na/tot*100:4.1f}%)  ← sub_total 이 없어서 검산을 '못' 한다")
print(f"  성립        {ok:3}행 ({ok/tot*100:4.1f}%)")
print(f"  ★ 불성립    {bad:3}행 ({bad/(ok+bad)*100:4.1f}% of 적용가능)  ← 정답인데 검산이 실패한다")

print("\n불성립 행의 배율 (품목합 / subtotal)")
print(" ", dict(collections.Counter(ratios).most_common()))

print("\n허용오차를 넓히면 잡히는가")
for tol in (0.0, 0.005, 0.01, 0.05, 0.10):
    o, b, _, _ = check_subtotal(P3, tol=tol)
    print(f"  ±{tol*100:5.1f}%   성립률 {o/(o+b)*100:5.1f}%   (성립 {o} / 불성립 {b})")

전체 200행
  적용 불가    69행 (34.5%)  ← sub_total 이 없어서 검산을 '못' 한다
  성립        122행 (61.0%)
  ★ 불성립      9행 ( 6.9% of 적용가능)  ← 정답인데 검산이 실패한다

불성립 행의 배율 (품목합 / subtotal)
  {1.1: 3, 2.0: 2, 0.942: 1, 1.57: 1, 1.531: 1, 1.667: 1}

허용오차를 넓히면 잡히는가
  ±  0.0%   성립률  92.4%   (성립 121 / 불성립 10)
  ±  0.5%   성립률  93.1%   (성립 122 / 불성립 9)
  ±  1.0%   성립률  93.1%   (성립 122 / 불성립 9)
  ±  5.0%   성립률  93.1%   (성립 122 / 불성립 9)
  ± 10.0%   성립률  95.4%   (성립 125 / 불성립 6)


### 읽는 법 — 두 가지가 나왔고 둘 다 설계를 바꾼다

```
전체 200행
  적용 불가    69행 (34.5%)   ← sub_total 이 없어서 검산을 '못' 한다
  성립        122행
  ★ 불성립      9행 (6.9% of 적용가능)   ← 정답인데 검산이 실패한다
```

### ① 합계 검산은 3분의 1에서 아예 작동하지 않는다

`sub_total` 이 없는 **67행** 때문이다. 이건 **틀린 게 아니라 못 하는** 것이고,
`CheckResult.passed = None` 로 정확히 구분해야 한다.
**적용 불가를 실패로 세면 검출력이 과소평가되고, 통과로 세면 과대평가된다.**

→ 검산 검출력을 보고할 때는 **반드시 "적용 가능 행 수" 를 함께 적는다.**
`md-rag-chatbot` 에서 지표 때문에 여섯 번 틀린 것 중 셋이 이런 종류였다.

### ② 정답 라벨에서도 6.9% 가 실패한다 = 헛경보의 바닥

초안 가설(*"검산 실패가 하나라도 있으면 보류"*)을 그대로 쓰면
**모델이 완벽하게 읽어도 6.9% 를 사람에게 넘긴다.** 그건 모델 탓이 아니라 문서 탓이다.

> ### ⚠️ 이 문단은 `0-8` 에서 뒤집혔다
> *"문서 탓"* 이 아니라 **우리 검산 규칙이 미비했던 것**이다.
> 불성립 6행을 하나씩 열어 원인 넷을 규칙으로 표현하니 **6.9% → 0%** 가 됐다.
> 지우지 않고 남긴다 — **"바닥" 이라고 이름 붙인 숫자가 사실은 내 코드의 한계였던** 사례다.

### ★ 그리고 허용 오차로는 못 잡는다 — 가설 하나가 죽었다

`code_design.md` 의 실험① 질문:

> 검산 ①(칼로리 환산)의 허용 오차는 몇 % 인가 — **반올림 때문에 정확히 안 맞는다**

**재보니 반올림이 원인이 아니다.** 그런데 여기서 **내가 먼저 쓴 결론이 실행에 반박당했다.**

```
± 0.0%   92.4%
± 0.5%   93.1%
± 1.0%   93.1%     ← 0.5% 에서 5% 까지, 10배를 넓혀도 한 행도 안 는다
± 5.0%   93.1%
±10.0%   95.4%     ← 3행이 새로 통과한다
```

처음엔 5% 까지만 재고 *"허용 오차로는 못 잡는다"* 고 적었다. **±10% 를 넣어보니 3행이 통과했다.**
그리고 그 3행이 정확히 **배율 `1.1` — 세금 10%** 다.

> ### 통과했다고 고쳐진 게 아니다
> `±10%` 허용은 **반올림을 봐주는 게 아니라 세금 한 겹을 통째로 눈감는 것**이다.
> 그러면 VLM 이 진짜로 10% 를 잘못 읽었을 때도 같이 통과한다.
>
> `0-1` 의 원래 걱정 — *"정규화를 세게 하면 진짜 오류까지 덮는다"* — 이
> **허용 오차 축에서 그대로 재현됐다.** `0-5` 에서 정규화 축의 위험이 반대 방향이라는 걸
> 확인했는데, 이 축에서는 원래 걱정이 맞았다. **축마다 위험 방향이 다르다.**

불성립 배율을 보면 이유가 보인다 — `1.1`(세금 10%) · `2.0`(중복 계상) · `1.67` · `0.942`.
**구조적이지 오차가 아니다.** 오차라면 1 근처에 몰려야 하는데 흩어져 있다.

> **"허용 오차를 얼마로 할 것인가" 는 잘못된 질문이었다.**
> 옳은 질문은 **"어떤 구조에서 등식이 성립하지 않는가"** 다.
> 세금이 품목에 포함된 영수증, subtotal 의 정의가 다른 영수증 — 그건 임계값이 아니라 **분기**다.
> 임계값으로 덮으면 그 폭만큼 진짜 오류도 함께 통과한다.
>
> `answer-first-approach` 와 같은 전환이다. *"재순위를 빠르게"* 가 아니라
> *"재순위를 어디서 돌릴 것인가"* 였던 것처럼, 여기서도 튜닝할 상수가 아니라 나눌 경우다.
>
> → **잠정 채택: ±0.5%.** 5% 까지 넓혀도 이득이 0 이고, 10% 는 세금을 은폐한다.
>   근거는 이 표다. 상수에 실험 번호를 단다 (`RERANK_THRESHOLD` 와 같은 자리).

### 다음

- **Phase 1** 두 모델 기준선 — 이제 정답 라벨을 숫자로 신뢰할 수 있다 (P3 + `keep_sub=True`)
- **Phase 2** 교차검증 전제 검증 ← 이 노트북의 목적
- 불성립 9행의 구조를 분류해서 검산 규칙을 분기시킬 것인가 — Phase 3 에서

---

# 0-8. ★ "임계값이 아니라 분기다" 를 실제로 해보다

`0-7` 에서 이렇게 적었다.

> **"허용 오차를 얼마로 할 것인가" 는 잘못된 질문이었다.**
> 옳은 질문은 **"어떤 구조에서 등식이 성립하지 않는가"** 다. 임계값이 아니라 **분기**다.

말은 했는데 **분기가 무엇인지는 안 적었다.** 이 절이 그걸 채운다.

## 먼저 이미지를 봤다

숫자만 보다가 막혀서 영수증 사진을 열었다 (행 85).

```
 3 Nasi Putih          30.000
 1 Nasi Tambah          7.000
 ...
 Subtotal            179.500
 Pajak Rest.10%       17.950     ← ★ 여기
14 Total             197.450
```

**`Pajak Rest. 10%` — 식당세 10%.** 배율 `1.1` 의 정체다.
그런데 이 영수증은 검산이 **성립한다** (품목합 179,500 = subtotal 179,500).
세금이 subtotal **밖에** 있기 때문이다.

불성립한 행들을 다시 보니 반대였다.

| | 품목합 | subtotal | tax | |
|---|---|---|---|---|
| 행 85 | 179,500 | **179,500** | 17,950 | 세금이 밖 → 성립 |
| 행 48 | 25,000 | 22,728 | 2,272 | **22,728 + 2,272 = 25,000** |
| 행 102 | 20,000 | 18,181 | 1,818 | **18,181 + 1,818 = 20,000** |
| 행 105 | 8,000 | 7,273 | 727 | **7,273 + 727 = 8,000** |

**같은 데이터셋 안에 세금 표기 관행이 두 가지다.**

```
관행 A   품목가가 세전   →   Σ품목 = subtotal          (세금은 subtotal 밖)
관행 B   품목가에 세금 포함 →   Σ품목 = subtotal + tax    (subtotal 이 세전 금액)
```

인도네시아 PPN 10% 가 품목 가격에 이미 녹아 있는 영수증들이다.
**검산 규칙이 하나면 두 관행 중 하나는 항상 틀린다.**

## 그리고 원인이 셋 더 나왔다 — 전부 우리 코드 문제였다

불성립 행을 하나씩 열어보니, "라벨이 이상하다" 고 넘길 뻔한 것들이 **우리 평탄화의 결함**이었다.

**① `sub` 가 하위 품목이 아닐 때가 있다** (행 186)

```json
{"nm": "LEMONADE22OZ", "cnt": "1", "price": "26,000",
 "sub": {"price": "26,000"}}                      ← 이름이 없다
```

`nm` 없이 금액만 있는 `sub` 는 **하위 품목이 아니라 금액 재인쇄**다.
영수증에 같은 금액이 두 번 찍힌 것을 라벨이 그대로 담았다. 펼치면 **이중 계상**된다.
→ **`nm` 이 있는 `sub` 만 품목으로 센다.**

**② `price` 가 없는 품목이 있다** (행 26)

```json
{"nm": "Teh Tawar", "unitprice": "4.000", "cnt": "2"}    ← price 없음
```

130,000 + (4,000 × 2) = **138,000 = subtotal.** 정확히 맞는다.
→ **`price` 가 없으면 `unitprice × cnt` 로 보완한다.**

**③ 품목 단위 할인이 있다** (행 31)

```json
[{"nm": "SALMON PESTO", "cnt": "1", "price": "85,500"},
 {"nm": "SALMON PESTO", "cnt": "1", "price": "85,500",
  "discountprice": "-85,500"}]                    ← 1+1 프로모션
```

품목은 진짜 둘인데 하나가 전액 할인. `85,500 × 2 - 85,500 = 85,500 = subtotal.`
→ **품목의 `discountprice` 를 더한다** (음수로 들어온다).

**④ 그런데 ③ 을 항상 적용하면 이중 계상된다** (행 107 · 196)

```json
menu       {"nm": "Choco Devil", "price": "63,636", "discountprice": "-9,545"}
sub_total  {"subtotal_price": "63,636", "discount_price": "-9,545", ...}
                                         ↑ 소계 쪽에도 같은 할인이 있다
```

`subtotal_price` 가 **할인 전** 금액이다. 여기서 품목 할인을 또 빼면 두 번 빠진다.
→ **`sub_total` 에 `discount_price` 가 있으면 품목 할인을 적용하지 않는다.**

★ ③ 과 ④ 는 서로 반대 방향이고, **`sub_total` 을 보면 어느 쪽인지 결정된다.**
추측이 아니라 데이터에서 판별 가능한 분기다.

In [9]:
# ── 규칙을 하나씩 쌓으며 기여도를 잰다 (ablation) ──────────
def line_total(it, r_qty):
    """품목 한 줄의 금액. r_qty=True 면 price 없을 때 unitprice × cnt 로 보완"""
    if "price" in it:
        return P3(it["price"])
    if r_qty and "unitprice" in it and "cnt" in it:
        u, c = P3(it["unitprice"]), P3(it["cnt"])
        if u is not None and c is not None:
            return u * c
    return None


def sum_items(gt, r_sub=False, r_qty=False, r_disc=False, r_dedup=False):
    """Σ품목. 하나라도 못 읽으면 None(적용 불가) — 0 으로 때우지 않는다"""
    st = gt.get("sub_total")
    # ④ 소계에 discount_price 가 따로 있으면 품목 할인은 소계 밖에서 처리된 것
    skip_disc = r_dedup and isinstance(st, dict) and "discount_price" in st
    vals = []
    for it in as_list(gt.get("menu")):
        if not isinstance(it, dict):
            continue
        v = line_total(it, r_qty)
        if v is None:
            return None
        if r_disc and not skip_disc and "discountprice" in it:   # ③ 품목 단위 할인
            dv = P3(it["discountprice"])
            if dv is not None:
                v += dv
        vals.append(v)
        for s in as_list(it.get("sub")):
            if not isinstance(s, dict):
                continue
            if r_sub and "nm" not in s:      # ① 이름 없는 sub 는 금액 재인쇄다
                continue
            sv = line_total(s, r_qty)
            if sv is not None:
                vals.append(sv)
    return sum(vals) if vals else None


def evaluate(tol=0.005, r_tax=False, **rules):
    """정답 라벨에서 Σ품목 ≈ subtotal 이 성립하는가"""
    ok = bad = na = 0
    fails = []
    for i, g in enumerate(GT):
        st = g.get("sub_total")
        sv = P3(st.get("subtotal_price")) if isinstance(st, dict) else None
        s = sum_items(g, **rules)
        if sv is None or sv == 0 or s is None:
            na += 1
            continue
        tax = P3(st.get("tax_price")) if "tax_price" in st else None
        hit = abs(s - sv) / abs(sv) <= tol
        if not hit and r_tax and tax is not None:      # ② 관행 B — 품목가에 세금 포함
            hit = abs(s - (sv + tax)) / abs(sv + tax) <= tol
        if hit:
            ok += 1
        else:
            bad += 1
            fails.append((i, round(s / sv, 3)))
    return ok, bad, na, fails


STEPS = [
    ("기준 (규칙 없음)",                   {}),
    ("+ ① sub 는 nm 있을 때만",            dict(r_sub=True)),
    ("+ ② price 결측 → unitprice×cnt",    dict(r_sub=True, r_qty=True)),
    ("+ ③ discountprice 반영",            dict(r_sub=True, r_qty=True, r_disc=True)),
    ("+ ④ 세금 분기 Σ≈subtotal+tax",       dict(r_sub=True, r_qty=True, r_disc=True, r_tax=True)),
    ("+ ⑤ 할인 이중계상 방지",              dict(r_sub=True, r_qty=True, r_disc=True, r_tax=True, r_dedup=True)),
]
print(f"{'누적 규칙':36} {'성립':>4} {'불성립':>6} {'적용불가':>8}  성립률   변화")
prev = None
for name, kw in STEPS:
    ok, bad, na, fails = evaluate(**kw)
    rate = ok / (ok + bad) * 100
    print(f"{name:36} {ok:4} {bad:6} {na:8}  {rate:5.1f}%"
          + ("" if prev is None else f"  {rate-prev:+5.1f}%p"))
    prev = rate

ok, bad, na, fails = evaluate(**STEPS[-1][1])
print(f"\n남은 불성립: {fails or '없음'}")

print("\n분기 대신 허용오차를 넓혔다면 (규칙 없음)")
for tol in (0.005, 0.05, 0.10):
    o, b, _, _ = evaluate(tol=tol)
    print(f"  ±{tol*100:4.1f}%   성립률 {o/(o+b)*100:5.1f}%")

누적 규칙                                  성립    불성립     적용불가  성립률   변화
기준 (규칙 없음)                            121      8       71   93.8%
+ ① sub 는 nm 있을 때만                    122      7       71   94.6%   +0.8%p
+ ② price 결측 → unitprice×cnt          124      7       69   94.7%   +0.1%p
+ ③ discountprice 반영                  126      5       69   96.2%   +1.5%p
+ ④ 세금 분기 Σ≈subtotal+tax              129      2       69   98.5%   +2.3%p
+ ⑤ 할인 이중계상 방지                        131      0       69  100.0%   +1.5%p

남은 불성립: 없음

분기 대신 허용오차를 넓혔다면 (규칙 없음)
  ± 0.5%   성립률  93.8%
  ± 5.0%   성립률  93.8%
  ±10.0%   성립률  95.3%


## 결과 — 그리고 `0-7` 의 결론 하나가 뒤집혔다

```
누적 규칙                              성립  불성립  적용불가  성립률   변화
기준 (규칙 없음)                        121     8      71    93.8%
+ ① sub 는 nm 있을 때만                 122     7      71    94.6%   +0.8%p
+ ② price 결측 → unitprice×cnt         124     7      69    94.7%   +0.1%p
+ ③ discountprice 반영                 126     5      69    96.2%   +1.5%p
+ ④ 세금 분기 Σ≈subtotal+tax            129     2      69    98.5%   +2.3%p
+ ⑤ 할인 이중계상 방지                    131     0      69   100.0%   +1.5%p
```

**131 / 131 = 100%.** 정답 라벨에서 합계 검산이 **한 행도 빠짐없이 성립한다.**

### 임계값 vs 분기 — 같은 문제, 다른 답

```
분기 5개  · 허용오차 ±0.5%   →  100.0%
허용오차만 ±10%              →   95.3%     그리고 10% 오독을 전부 통과시킨다
```

**분기가 더 많이 잡으면서 오차 허용은 20배 좁다.**
±10% 는 3행을 살리려고 **모든 행에서 10% 오독을 눈감는** 거래였다.
분기는 87행(tax 있는 행)에서만, 그것도 **특정 구조에만** 예외를 준다.

### ★ "6.9% 헛경보 바닥" 은 바닥이 아니었다

`0-7` 에 이렇게 적었다.

> **정답 라벨에서도 6.9% 가 실패한다 = 헛경보의 바닥**
> 초안 가설을 그대로 쓰면 **모델이 완벽하게 읽어도 6.9% 를 사람에게 넘긴다.**
> 그건 모델 탓이 아니라 **문서 탓**이다.

**틀렸다. 문서 탓이 아니라 우리 규칙이 미비했던 것이다.** 규칙 다섯을 채우니 0% 가 됐다.

> ### 여기서 배운 것
> **"상한" 이나 "바닥" 이라고 이름 붙인 숫자를 조심한다.**
> 그건 데이터의 성질일 수도 있고 **내 코드의 한계일 수도 있다.**
> 6.9% 를 바닥으로 받아들였다면 `judge()` 에 *"헛경보 7% 는 어쩔 수 없다"* 는
> 상수가 근거처럼 박혔을 것이다.
>
> 구분하는 방법은 하나뿐이다 — **불성립 행을 하나씩 열어보는 것.**
> 6행을 열었더니 원인이 넷이었고 전부 규칙으로 표현 가능했다.
> `0-6` 에서 *"검산 통과는 반증 실패다"* 라고 했는데, **검산 실패도 마찬가지다** —
> 데이터가 틀렸다는 증거가 아니라 **내 규칙이 이 구조를 모른다는 증거**일 수 있다.

### 남은 것 — 적용 불가 69행(34.5%)

여기는 변하지 않았다. `sub_total` 이 아예 없어서 **검산을 못 하는** 행이다.
`total_price` 로 대체 검산이 가능한지는 Phase 3 에서 본다.

**검출력을 보고할 때는 항상 분모를 함께 적는다** — `131행 중` 이지 `200행 중` 이 아니다.

---

# Phase 1 — 모델 하나로 어디까지 가는가

`0-1` 정정에서 본 대로 **무료 티어가 실험 범위를 정했다.**

| 한도 | 값 | 어디에 적혀 있나 |
|---|---|---|
| TPM (분당 토큰) | 8,000 | 응답 헤더 |
| ITPM (분당 입력) | 7,000 | **429 본문** |
| RPD (하루 요청) | 1,000 | 응답 헤더 |
| **TPD (하루 토큰)** | **200,000** | **429 본문에만** |

```
CORD 이미지 1장 ≈ 2,200 토큰  →  200,000 ÷ 2,200 ≈ 하루 90장 / 모델
```

**TPD 가 실질 상한이고 TPM 은 그 안에서의 속도다.**
헤더가 TPM·RPD 두 개를 보여주길래 그게 전부라고 읽었는데, **가장 아픈 것이 429 본문에만 있었다.**
잔여요청 966/1000 을 보고 *"무료 안에서 확실히 끝난다"* 고 단언했다가 41행에서 멈췄다.

> ### 같은 실수의 세 번째다
> | | 무엇을 | 어떻게 틀렸나 |
> |---|---|---|
> | ① | SROIE = MIT | 미러(코드 저장소) 표기를 **원본 데이터셋의 것**으로 읽음 |
> | ② | Gemini 1,500 요청/일 | **문서에서 읽은 숫자**를 실측 옆에 나란히 적음 (실제 20) |
> | ③ | Groq 한도 = 헤더가 전부 | **헤더에 없는 것을 없다고** 가정 (TPD 200k) |
>
> ①은 **대상**을 안 봤고, ②는 **출처**를 안 봤고, ③은 **없는 것을 봤다**고 했다.
> **한도는 문서·응답 헤더·429 본문 세 곳에 흩어져 있다.**

### 그래서 이번 표본은 41행이다

| | 모델 | 행 | 역할 |
|---|---|---|---|
| 본 측정 | `qwen/qwen3.8-27b` | **0~40 (41행)** | Phase 1 기준선 |
| 대조군 | `qwen/qwen3.6-27b` | **0~40 (41행)** | 같은 계열 짝 |

둘이 **같은 41행**이라 짝이 맞는다. 분모를 같게 만든 것은 의도한 것이다 —
`md-rag-chatbot` 에서 지표로 틀린 셋 중 하나가 분모를 안 적어서였다.

**★ 표본 41행은 작다.** 아래 모든 숫자에 오차 건수를 함께 적는다.
`3/6` 같은 값은 비율보다 **분자·분모 자체**를 봐야 한다.

In [10]:
import pickle, collections

CACHE_DIR = DATA
def load_calls(model):
    """배치 러너가 남긴 호출 캐시. 성공한 호출만 담겨 있다.
       ★ 실패를 캐시했다가 재시도가 영영 막힌 적이 있다 — 캐시는 성공만 담는다"""
    f = CACHE_DIR / f"calls_{model.replace('/','_')}.pkl"
    if not f.exists():
        return {}
    return {v["row"]: v for v in pickle.loads(f.read_bytes()).values()}

MODEL_MAIN, MODEL_CTRL = "qwen/qwen3.8-27b", "qwen/qwen3.6-27b"
A, B = load_calls(MODEL_MAIN), load_calls(MODEL_CTRL)
ROWS = sorted(set(A) & set(B))        # 짝이 맞는 행만 쓴다
print(f"본 측정 {len(A)}행 · 대조군 {len(B)}행 · 공통 {len(ROWS)}행 (행 {min(ROWS)}~{max(ROWS)})")

print("\n[1] 스키마 준수 — Groq 어댑터의 비대칭이 실제로 어땠나")
for name, C in ((MODEL_MAIN, A), (MODEL_CTRL, B)):
    n = len(C)
    fenced = sum(bool(v.get("fenced")) for v in C.values())
    think  = sum(bool(v.get("think"))  for v in C.values())
    unpar  = sum(not v.get("parsed")   for v in C.values())
    notdict= sum(not isinstance(v.get("data"), dict) for v in C.values())
    print(f"  {name}")
    print(f"    ```json 펜스로 감쌈   {fenced:3}/{n}  ({fenced/n*100:3.0f}%)")
    print(f"    <think> 블록          {think:3}/{n}  ({think/n*100:3.0f}%)")
    print(f"    JSON 파싱 실패        {unpar:3}/{n}  ({unpar/n*100:3.0f}%)")
    print(f"    최상위가 dict 아님     {notdict:3}/{n}  ({notdict/n*100:3.0f}%)")

본 측정 41행 · 대조군 41행 · 공통 41행 (행 0~40)

[1] 스키마 준수 — Groq 어댑터의 비대칭이 실제로 어땠나
  qwen/qwen3.8-27b
    ```json 펜스로 감쌈    41/41  (100%)
    <think> 블록            0/41  (  0%)
    JSON 파싱 실패          0/41  (  0%)
    최상위가 dict 아님       0/41  (  0%)
  qwen/qwen3.6-27b
    ```json 펜스로 감쌈    41/41  (100%)
    <think> 블록           41/41  (100%)
    JSON 파싱 실패          0/41  (  0%)
    최상위가 dict 아님       1/41  (  2%)


### [1] 읽는 법 — "스키마를 안 지킨다" 는 한 가지가 아니다

`2026-09-06` 기록에 이렇게 적어뒀다.

> Gemini 는 Pydantic 모델을 넘기면 출력을 스키마로 강제한다 → 파싱 코드 불필요
> Groq 는 ` ```json … ``` ` 마크다운 펜스로 감싼 텍스트를 뱉는다 → 파싱 + 검증 필요

**실측하니 위반이 세 층으로 갈렸다.**

| 층 | 무엇 | 대응 |
|---|---|---|
| 포장 | ` ```json ` 펜스 | 정규식으로 벗긴다 |
| 사고 과정 | `<think> … </think>` (qwen3.6 만) | 펜스 벗기기 **전에** 제거 |
| 구조 | 최상위가 `dict` 가 아니라 `list` | 스키마 위반. 세어서 지표로 쓴다 |

앞의 둘은 **포장**이라 벗기면 되고, 세 번째는 **진짜 위반**이다.
*"모델 B 가 스키마를 안 지켜서 실패한 비율"* 을 지표로 쓰겠다고 적어놨는데,
**한 숫자로 뭉치면 안 된다** — 벗길 수 있는 것과 못 쓰는 것을 나눠 세야 한다.

In [11]:
# ── 모델 출력 vs 정답 비교 ────────────────────────────────
# ★ CORD 는 합계를 여러 키로 쓴다. total_price 가 없으면 동의어를 후보로 본다.
#   이걸 안 하면 "라벨 스키마 차이" 를 "모델 오류" 로 세게 된다 (아래 [3] 참고)
ALIAS = {"total_price":    ["total_price", "creditcardprice", "emoneyprice", "total_etc"],
         "subtotal_price": ["subtotal_price"]}

def mget(m, k):
    return m.get(k) if isinstance(m, dict) else None

def out_items(m):
    if not isinstance(m, dict): return []
    return [x for x in (m.get("menu") or []) if isinstance(x, dict)]

def norm_name(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower()) if s else ""

def gt_menu(g):
    out = []
    for it in as_list(g.get("menu")):
        if not isinstance(it, dict): continue
        out.append(it)
        for s in as_list(it.get("sub")):
            if isinstance(s, dict) and "nm" in s: out.append(s)
    return out

def compare(i, C):
    """필드별 일치 여부. True / False / "n/a"(라벨에 근거가 없어 채점 불가)"""
    r = C.get(i)
    if not r or not r.get("parsed") or not isinstance(r["data"], dict): return None
    m, g = r["data"], GT[i]
    gi, mi = gt_menu(g), out_items(m)
    o = {"n_items": len(gi) == len(mi)}
    o["names"]  = collections.Counter(norm_name(x.get("nm")) for x in gi) == \
                  collections.Counter(norm_name(x.get("nm")) for x in mi)
    o["prices"] = collections.Counter(P3(x.get("price")) for x in gi) == \
                  collections.Counter(P3(x.get("price")) for x in mi)
    o["cnts"]   = collections.Counter(P3(x.get("cnt")) for x in gi) == \
                  collections.Counter(P3(x.get("cnt")) for x in mi)
    for key, sec in (("subtotal_price", "sub_total"), ("total_price", "total")):
        d0 = g.get(sec) if isinstance(g.get(sec), dict) else {}
        cands = [v for v in (P3(d0[k]) for k in ALIAS[key] if k in d0) if v is not None]
        mv = P3(mget(m, key))
        o[key] = "n/a" if not cands else (mv is not None and any(abs(c - mv) < 0.01 for c in cands))
    return o

FIELDS = ["n_items", "names", "prices", "cnts", "subtotal_price", "total_price"]
MONEY  = ["prices", "subtotal_price", "total_price"]
res = {i: compare(i, A) for i in ROWS}
res = {i: c for i, c in res.items() if c}

print(f"[2] 필드별 정확도 — {MODEL_MAIN} · {len(res)}행")
for f in FIELDS:
    v = [c[f] for c in res.values() if c[f] != "n/a"]
    print(f"    {f:16} {sum(v):3}/{len(v):3}   {sum(v)/len(v)*100:5.1f}%")

def all_ok(c, keys):  return all(c[f] is True for f in keys if c[f] != "n/a")
strict = sum(all_ok(c, FIELDS) for c in res.values())
money  = sum(all_ok(c, MONEY)  for c in res.values())
print(f"\n[3] 행 단위")
print(f"    전 필드 완전 일치   {strict:3}/{len(res)}   {strict/len(res)*100:5.1f}%")
print(f"    금액 3종만 일치     {money:3}/{len(res)}   {money/len(res)*100:5.1f}%")
why = collections.Counter()
for c in res.values():
    for f in FIELDS:
        if c[f] is False: why[f] += 1
print("    틀린 필드 분포:", dict(why.most_common()))

[2] 필드별 정확도 — qwen/qwen3.8-27b · 41행
    n_items           35/ 41    85.4%
    names             25/ 41    61.0%
    prices            32/ 41    78.0%
    cnts              31/ 41    75.6%
    subtotal_price    25/ 26    96.2%
    total_price       40/ 40   100.0%

[3] 행 단위
    전 필드 완전 일치    21/41    51.2%
    금액 3종만 일치      32/41    78.0%
    틀린 필드 분포: {'names': 16, 'cnts': 10, 'prices': 9, 'n_items': 6, 'subtotal_price': 1}


### [2][3] ★ 지표를 한 번 고치니 거의 모든 숫자가 움직였다

처음엔 *"정답 라벨에 값이 없는데 모델이 값을 냈다"* 를 **오답**으로 셌다. 그런데 열어보니:

```
행  9   정답 total: {cashprice 50000, changeprice 14000}    모델 total_price 36,000
        →  50,000 − 14,000 = 36,000.  ★ 모델이 맞다. 라벨에 total_price 키가 없을 뿐
행 37   정답 total_etc '20.000'                             모델 20,000    같은 값, 키 이름만 다름
행 39   정답 creditcardprice '3,112,800'                    모델 3,112,800  같은 값
```

**CORD 는 합계를 `total_price` · `creditcardprice` · `emoneyprice` · `total_etc` 로 나눠 쓴다.**
우리 스키마는 `total_price` 하나뿐이라, 라벨 키가 다르면 모델이 맞아도 오답이 됐다.

동의어를 후보로 인정하고 **라벨에 근거가 없으면 `n/a`(채점 불가)** 로 뺐더니:

| | 고치기 전 | 고친 뒤 |
|---|---|---|
| `total_price` | 92.7% | **100.0%** |
| `subtotal_price` | 80.6% | **96.2%** |
| 행 완전 일치 | 43.9% | 51.2% |
| **금액만** | 63.4% | **78.0%** |
| **검산 통과분 정확도** | 72% | **88%** |

> **지표 하나를 고쳤을 뿐인데 여섯 숫자가 전부 움직였다.**
> `md-rag-chatbot` 실험에서 지표 때문에 여섯 번 틀렸고 **그중 셋이 스스로 만든 지표**였다.
> 여기서 또 한 번 만들었다. **모델이 틀린 게 아니라 내 채점표가 틀렸다.**
>
> 구분법은 `0-8` 과 같다 — **틀렸다고 나온 행을 하나씩 열어보는 것.**

### 그리고 지표 정의가 20%p 를 좌우한다

```
전 필드 기준   21/41 = 51.2%      품목명 오타 하나로도 '틀림'
금액만 기준    32/41 = 78.0%      숫자만 본다
```

**어느 쪽도 틀리지 않았고, 무엇을 지키려는지에 따라 답이 다르다.**
이 엔진이 지키려는 것은 *"뽑은 **숫자**를 믿어도 되는가"* 이므로 **금액 기준을 주 지표로 삼는다.**
`0-1` 의 숙제 ②(*필드마다 성격이 다른데 어떻게 합치나*)에 대한 답이다 —
**합치지 않고, 무엇을 지키는지 먼저 정하고 그 필드만 본다.**

### 필드별로 보면 실패가 한쪽에 쏠려 있다

```
total_price     100.0%    ← 합계는 완벽하게 읽는다
subtotal_price   96.2%
n_items          85.4%
prices           78.0%
cnts             75.6%
names            61.0%    ← 품목명이 최악
```

**합계는 완벽한데 품목명은 61% 다.** 큰 글씨로 한 줄에 있는 값은 잘 읽고,
여러 줄에 걸친 목록에서 이름-금액 짝을 맞추는 데서 무너진다.
(`0-8` 에서 본 영수증의 `Sambal Gabus Asin Cb` 처럼 이름이 길어 금액이 다음 줄로 밀린 자리)

In [12]:
# ── [4] ★ 검산 검출력 ─────────────────────────────────────
def check_output(m, tol=0.005):
    """모델이 뱉은 JSON 안에서 Σ품목 ≈ subtotal 이 성립하는가.
       ★ 정답을 보지 않는다 — 실제 운영에서 쓸 수 있는 유일한 신호다.
       True(통과) / False(실패) / None(적용 불가)"""
    sv = P3(mget(m, "subtotal_price"))
    if sv is None or sv == 0: return None
    ps = []
    for it in out_items(m):
        v = P3(it.get("price"))
        if v is None and it.get("unitprice") and it.get("cnt"):
            u, c = P3(it["unitprice"]), P3(it["cnt"])
            v = u * c if (u is not None and c is not None) else None
        if v is None: return None
        ps.append(v)
    if not ps: return None
    return abs(sum(ps) - sv) / abs(sv) <= tol


tab = collections.Counter(); na = 0
for i, c in res.items():
    v = check_output(A[i]["data"])
    if v is None: na += 1; continue
    tab[(v, all_ok(c, MONEY))] += 1          # 실제 정답 여부는 '금액 기준'
tn, fn = tab[(True, True)],  tab[(True, False)]
fp, tp = tab[(False, True)], tab[(False, False)]

print("[4] 검산 검출력 — 모델 출력만 보고 판정할 수 있나")
print(f"    적용 가능 {tp+fp+fn+tn}행 · 적용 불가 {na}행 (모델이 subtotal 을 안 냄)\n")
print("                    실제 맞음   실제 틀림")
print(f"      검산 통과    {tn:8}   {fn:8}   ← 못 잡는다")
print(f"      검산 실패    {fp:8}   {tp:8}   ← 잡았다")
print(f"\n    재현율  {tp}/{tp+fn} = {tp/(tp+fn)*100:.0f}%   틀린 것 중 잡은 비율")
print(f"    정밀도  {tp}/{tp+fp} = {tp/(tp+fp)*100:.0f}%   경보 중 진짜 비율")
print(f"    ★ 검산 통과분의 실제 정확도  {tn}/{tn+fn} = {tn/(tn+fn)*100:.0f}%")

# ── [5] 대조군 — 같은 계열 두 모델 ────────────────────────
print(f"\n[5] 같은 계열 대조군 — {MODEL_MAIN} vs {MODEL_CTRL} · {len(ROWS)}행")
dis = collections.Counter()
for i in ROWS:
    fa, fb = A[i].get("data"), B[i].get("data")
    if not (isinstance(fa, dict) and isinstance(fb, dict)):
        dis["한쪽이 dict 가 아님"] += 1; continue
    dis["total 불일치"]    += P3(fa.get("total_price"))    != P3(fb.get("total_price"))
    dis["subtotal 불일치"] += P3(fa.get("subtotal_price")) != P3(fb.get("subtotal_price"))
    dis["품목수 불일치"]     += len(out_items(fa)) != len(out_items(fb))
for k, v in dis.items():
    print(f"    {k:20} {v:3}/{len(ROWS)}  ({v/len(ROWS)*100:5.1f}%)")

# ── [6] Phase 2 의 2×2 를 같은 계열 짝으로 채운다 ──────────
print("\n[6] 교차검증 전제의 2×2 (같은 계열 짝)")
for field, sec in (("total_price", "total"), ("subtotal_price", "sub_total")):
    t2 = collections.Counter()
    for i in ROWS:
        fa, fb = A[i].get("data"), B[i].get("data")
        if not (isinstance(fa, dict) and isinstance(fb, dict)): continue
        d0 = GT[i].get(sec) if isinstance(GT[i].get(sec), dict) else {}
        cands = [v for v in (P3(d0[k]) for k in ALIAS[field] if k in d0) if v is not None]
        if not cands: continue
        va, vb = P3(fa.get(field)), P3(fb.get(field))
        t2[(va == vb, va is not None and any(abs(c - va) < 0.01 for c in cands))] += 1
    n = sum(t2.values())
    if not n: continue
    print(f"\n    [{field}] 정답 있는 {n}행        실제 맞음   실제 틀림")
    print(f"      두 모델 일치              {t2[(True,True)]:8}   {t2[(True,False)]:8}  ← 못 잡는다")
    print(f"      두 모델 불일치            {t2[(False,True)]:8}   {t2[(False,False)]:8}  ← 잡았다")

[4] 검산 검출력 — 모델 출력만 보고 판정할 수 있나
    적용 가능 29행 · 적용 불가 12행 (모델이 subtotal 을 안 냄)

                    실제 맞음   실제 틀림
      검산 통과          22          3   ← 못 잡는다
      검산 실패           1          3   ← 잡았다

    재현율  3/6 = 50%   틀린 것 중 잡은 비율
    정밀도  3/4 = 75%   경보 중 진짜 비율
    ★ 검산 통과분의 실제 정확도  22/25 = 88%

[5] 같은 계열 대조군 — qwen/qwen3.8-27b vs qwen/qwen3.6-27b · 41행
    total 불일치              0/41  (  0.0%)
    subtotal 불일치          12/41  ( 29.3%)
    품목수 불일치                5/41  ( 12.2%)
    한쪽이 dict 가 아님          1/41  (  2.4%)

[6] 교차검증 전제의 2×2 (같은 계열 짝)

    [total_price] 정답 있는 39행        실제 맞음   실제 틀림
      두 모델 일치                    39          0  ← 못 잡는다
      두 모델 불일치                   0          0  ← 잡았다

    [subtotal_price] 정답 있는 25행        실제 맞음   실제 틀림
      두 모델 일치                    24          0  ← 못 잡는다
      두 모델 불일치                   0          1  ← 잡았다


## [4] ★ 검산은 **정밀하지만 성기다**

```
                    실제 맞음   실제 틀림
      검산 통과          22          3     ← 못 잡는다
      검산 실패           1          3     ← 잡았다

    재현율  3/6  = 50%     틀린 것의 절반만 잡는다
    정밀도  3/4  = 75%     경보 4건 중 3건이 진짜
    ★ 검산 통과분의 실제 정확도  22/25 = 88%
```

**마지막 줄이 이 프로젝트의 답이다.** *"검산을 통과했으니 자동처리한다"* 로 가면 **12% 가 틀린 채 통과한다.**

> ### `0-6` 의 명제가 모델 출력에서 숫자로 확인됐다
> `0-6` 에서 파서 P2 가 47개 값에서 틀렸는데도 검산 성립률이 정답 파서와 같았다.
> 거기서 **"검산 통과는 근거가 아니라 반증 실패다"** 라고 썼다.
>
> 모델 출력에서도 같다 — **통과한 25행 중 3행이 실제로 틀렸다.**
> 검산은 **틀린 것을 잡지, 맞은 것을 증명하지 않는다.**
>
> → `confidence.judge()` 를 **"검산 통과 = 자동처리"** 로 짜면 안 된다.
> 검산 실패는 **보류의 근거**로 쓸 수 있지만(정밀도 75%), 검산 통과는 **자동처리의 근거가 못 된다.**

### 그래서 교차검증이 필요한 이유가 오히려 실측으로 드러났다

검산 단독의 상한이 **88%** 다. 나머지 12% 를 잡으려면 **다른 종류의 신호**가 있어야 하고,
그게 원래 설계의 교차검증이다. **교차검증을 포기한 게 아니라, 왜 필요한지가 숫자로 나온 셈이다.**

**적용 불가 12행(29%)** 도 그대로 남는다 — 모델이 `subtotal` 을 안 내면 검산 자체를 못 한다.
`0-7` 에서 정답 라벨로 34.5% 였던 것과 비슷한 크기다.

---

## [5][6] 대조군 — 같은 계열은 신호를 거의 못 만든다

```
total 불일치        0/41  (  0.0%)   ← 41행 전부 같은 답
subtotal 불일치    12/41  ( 29.3%)
품목수 불일치        5/41  ( 12.2%)
```

**`total_price` 에서 두 모델이 41행 내내 한 번도 갈리지 않았다.** 그리고 2×2 는:

```
    [total_price] 정답 있는 39행     실제 맞음   실제 틀림
      두 모델 일치                      39          0
      두 모델 불일치                     0          0
```

**오류가 0 이라 교차검증이 잡을 것도 없었다.** `subtotal_price` 에서만 불일치 1건이 나왔고
그 1건이 실제 오답이었다 (1/1).

### 무엇을 말할 수 있고, 무엇을 말할 수 없나

**말할 수 있는 것** — `total_price` 에서 같은 계열 두 모델은 **완전히 같은 답**을 낸다.
버전이 달라도(3.6 vs 3.8) 갈리지 않는다. **교차검증의 짝으로 쓸 이유가 없다.**

**말할 수 없는 것** — *"일치했는데 둘 다 틀림"* 이 얼마나 되는지. 이 41행에서는 **오류가 0** 이라
그 칸을 채울 수 없었다. 설계의 상한은 **여전히 미지수**다.

> ### ★ 표본이 작다는 것을 숫자로 말한다
> 오차 건수가 `3/6`, `1/1`, `0/39` 다. **비율보다 분자를 봐야 하는 크기다.**
> `1/1 = 100%` 는 100% 가 아니라 **"한 건 있었고 맞혔다"** 이다.
> 41행은 TPD 200,000 이 정한 크기이지 내가 고른 크기가 아니다.

### 다음
- **내일** — 두 모델을 90행까지 늘린다 (TPD 가 하루 90장을 준다)
- **날 잡아** — Gemini 20행/일 누적. 이 대조군이 해석 기준이 된다:
  같은 계열이 `total` 에서 0% 갈렸으니, **다른 벤더가 몇 % 갈리는지**가 곧 벤더 분리의 값어치다
- `names` 61% 의 실패 구조 — 이름-금액 짝 맞추기가 어디서 깨지는지

---

# 2026-09-07 — 데이터셋 확정 + Phase 0 + Phase 1(41행)

**Phase 1 — 표본 41행 (TPD 200,000 이 정한 크기)**

| | |
|---|---|
| `total_price` 정확도 | **100%** (40/40) |
| `names` 정확도 | **61%** ← 이름-금액 짝 맞추기에서 무너진다 |
| 행 단위 (금액 기준) | **78%** · 전 필드 기준 51% — **지표 정의가 20%p 를 좌우** |
| ★ **검산 통과분의 실제 정확도** | **88%** (22/25) — *"통과 = 자동처리"* 로 가면 **12% 가 틀린 채 통과** |
| 검산 재현율 / 정밀도 | 50% (3/6) / 75% (3/4) — **정밀하지만 성기다** |
| 대조군 `total` 불일치 | **0/41** — 같은 계열은 교차검증 짝으로 쓸 이유가 없다 |

**★ 지표를 한 번 고치니 여섯 숫자가 전부 움직였다** (`total_price` 92.7%→100%).
CORD 가 합계를 `creditcardprice`·`total_etc` 등으로 나눠 쓰는데 우리 스키마는 하나뿐이라
**라벨 스키마 차이를 모델 오류로 세고 있었다.** 모델이 틀린 게 아니라 채점표가 틀렸다.

**★ 한도 실측** — TPM 8,000 / ITPM 7,000 / RPD 1,000 / **TPD 200,000(429 본문에만)**.
이미지 1장 ≈ 2,200 토큰 → **하루 90장/모델.** 200행은 애초에 불가능했다.

---

# 2026-09-07 — 데이터셋 확정 + Phase 0

**라이선스**
- 원본에서 대조 → `SROIE = MIT` 이 틀렸음을 확인 (`0-2b`). MIT 는 참가팀 **코드**의 것이었다
- **CORD 단독 채택.** CC BY 4.0 · 품목 있음 · SROIE 능력을 포함한다
- README 에 CC BY 4.0 인용 블록(저작자·출처·라이선스·변경여부) 추가

**실측으로 정한 것**
- 로드는 `split=` 이 아니라 `data_files=` — `split=` 은 train 까지 받는다 (`0-3`)
- 금액 파서 **P3**(마지막 구분자 뒤 자릿수) — 순진한 파서는 200행 중 18행을 1000배로 깨뜨린다
- `sub` 하위품목은 **합계에 넣는다** — 93.1% vs 90.1%

**가설 둘이 죽었다**
- ~~"정규화를 세게 하면 진짜 오류를 덮는다"~~ → **잘못 정규화하면 없던 오류를 만든다.** 방향이 반대
- ~~"검산 허용 오차를 몇 %로 할 것인가"~~ → **반올림이 원인이 아니다.** ±0.5%~±5% 가 동률이고,
  ±10% 에서 3행이 통과하는데 그건 **세금 10% 를 눈감는 것**이다. 잠정 채택 **±0.5%**

**★ 설계를 건드리는 숫자**
- 합계 검산은 **34.5% 에서 적용 불가** (`sub_total` 없음) → `passed: bool | None` 이 옳았다
- **검산은 파서를 증명하지 못한다** — P2 는 47개 값에서 틀렸는데 성립률은 P3 와 동률 (`0-6`)

**★ 분기 규칙 5개로 정답 라벨 검산이 100% 성립 (`0-8`)**
① 이름 없는 `sub` 는 금액 재인쇄 · ② `price` 결측은 `unitprice×cnt` ·
③ 품목 `discountprice` 반영 · ④ **세금 표기 관행이 두 가지** (`Σ≈subtotal` / `Σ≈subtotal+tax`) ·
⑤ 소계에 `discount_price` 가 있으면 품목 할인 미적용(이중 계상 방지)

```
분기 5개 · 허용오차 ±0.5%   →  100.0%
허용오차만 ±10%             →   95.3%  + 모든 행에서 10% 오독을 눈감는다
```

- ~~"정답 라벨에서도 6.9% 실패 = 헛경보 바닥"~~ → **바닥이 아니라 규칙 미비였다.**
  **"상한·바닥" 이라 이름 붙인 숫자가 내 코드의 한계일 수 있다** — 불성립 행을 열어봐야 안다

---

# 2026-09-06 — 여기까지

**한 일**
- 모델 두 개 확정 (`gemini-2.5-flash` · `qwen/qwen3.8-27b`) — 비전 지원을 **실측으로** 확인
- 한도 실측 — Groq 분당 토큰이 병목, 약 4장/분
- 지표 설계의 숙제 세 가지를 적어둠
- **SROIE 로는 검산을 못 잰다**는 것을 발견 (설계 후에)

**다음**
1. ~~SROIE 내려받기 + 라이선스 대조~~ → **끝. 대조 결과 SROIE 를 버리고 CORD 로 갔다 (`0-2b`)**
2. **CORD 내려받기** + 라벨 구조 파악 (4범주 30종)
3. Phase 0 지표 확정 — 정규화를 어디까지 할 것인가
4. Phase 1 두 모델 기준선
5. Phase 2 교차검증 전제 검증 ← 이 노트북의 목적

**사람이 해야 할 것**
- 영양성분표 **20~30장 촬영**. ★ CORD 로 바꾸면서 **Phase 3 을 막지 않게 됐다** —
  검산 검출력은 CORD 로 먼저 재고, 촬영분은 "직접 만든 평가셋" 축으로 나중에 붙인다.